# 1. Image Preparations

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3, Xception
from tensorflow.keras.applications.inception_v3 import preprocess_input as inc_preprocess
from tensorflow.keras.applications.xception import preprocess_input as xcep_preprocess
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
BASE_DIR = "/workspaces/MNIST-CNN-FPGA/dataset"
TRAIN_DIR = os.path.join(BASE_DIR, "seg_train")
TEST_DIR = os.path.join(BASE_DIR, "seg_test")
PRED_DIR = os.path.join(BASE_DIR, "seg_pred")

In [ ]:
IMG_SIZE = (299, 299)
BATCH_SIZE = 32

# Generator untuk ekstraksi fitur tanpa augmentasi acak per epoch (atau bisa aktifkan augmentasi offline)
# Untuk ekstraksi fitur ke SVM, shuffle wajib False agar urutan fitur sesuai dengan urutan label!
datagen_train = ImageDataGenerator(preprocessing_function=inception_preprocess)
datagen_test = ImageDataGenerator(preprocessing_function=inception_preprocess)

train_generator = datagen_train.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    shuffle=False  # WAJIB False agar label y_train sejajar dengan hasil predict()
)

test_generator = datagen_test.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    shuffle=False
)

y_train = train_generator.classes
y_test = test_generator.classes
class_names = list(train_generator.class_indices.keys())
print("Kelas ditemukan:", class_names)

In [ ]:
# 1. Stream 1: Inception V3 sampai intermediate layer 'mixed_8'
base_inception = InceptionV3(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
model_inception_mid = Model(
    inputs=base_inception.input,
    outputs=base_inception.get_layer('mixed_8').output
)

# 2. Stream 2: Xception sampai Global Average Pooling
base_xception = Xception(weights='imagenet', include_top=False, pooling='avg', input_shape=(299, 299, 3))

print("Output shape Inception mixed_8:", model_inception_mid.output_shape)
print("Output shape Xception avg_pool:", base_xception.output_shape)

In [ ]:
def extract_features(generator, model_stream1, model_stream2):
    generator.reset()
    feat1 = model_stream1.predict(generator, verbose=1)
    # Ratakan tensor konvolusi menjadi 2D: (jumlah_gambar, total_fitur)
    feat1_flat = feat1.reshape(feat1.shape[0], -1)
    
    generator.reset()
    feat2 = model_stream2.predict(generator, verbose=1)
    
    return feat1_flat, feat2

print("Mengekstrak fitur training...")
X_train_conv, X_train_avg = extract_features(train_generator, model_inception_mid, base_xception)

print("Mengekstrak fitur testing...")
X_test_conv, X_test_avg = extract_features(test_generator, model_inception_mid, base_xception)

In [ ]:
# 1. PCA hanya di-fit pada data training agar tidak terjadi data leakage!
pca = PCA(n_components=512, random_state=42)
X_train_conv_pca = pca.fit_transform(X_train_conv)
X_test_conv_pca = pca.transform(X_test_conv)

# 2. L2 Normalization masing-masing stream
X_train_conv_norm = normalize(X_train_conv_pca, norm='l2')
X_test_conv_norm = normalize(X_test_conv_pca, norm='l2')

X_train_avg_norm = normalize(X_train_avg, norm='l2')
X_test_avg_norm = normalize(X_test_avg, norm='l2')

# 3. Feature Fusion (Concatenation)
X_train_fused = np.hstack([X_train_conv_norm, X_train_avg_norm])
X_test_fused = np.hstack([X_test_conv_norm, X_test_avg_norm])

print("Dimensi fitur final setelah fusion:", X_train_fused.shape)

param_grid = {'C': [0.01, 0.1, 1, 10, 100]}
svm = LinearSVC(max_iter=5000, random_state=42)

grid = GridSearchCV(svm, param_grid, cv=3, n_jobs=-1, verbose=1)
grid.fit(X_train_fused, y_train)

best_svm = grid.best_estimator_
print("Parameter C terbaik:", grid.best_params_)

# Evaluasi pada seg_test
y_pred = best_svm.predict(X_test_fused)
acc = accuracy_score(y_test, y_pred)
print(f"Akurasi pada seg_test: {acc * 100:.2f}%\n")
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
import pandas as pd

pred_files = [f for f in os.listdir(PRED_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
df_pred = pd.DataFrame({'filename': pred_files})

pred_generator = datagen_test.flow_from_dataframe(
    dataframe=df_pred,
    directory=PRED_DIR,
    x_col='filename',
    y_col=None,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode=None,
    shuffle=False
)

# Ekstrak & transformasi fitur untuk seg_pred
X_pred_conv, X_pred_avg = extract_features(pred_generator, model_inception_mid, base_xception)
X_pred_conv_norm = normalize(pca.transform(X_pred_conv), norm='l2')
X_pred_avg_norm = normalize(X_pred_avg, norm='l2')
X_pred_fused = np.hstack([X_pred_conv_norm, X_pred_avg_norm])

# Prediksi kelas
pred_indices = best_svm.predict(X_pred_fused)
df_pred['predicted_scene'] = [class_names[i] for i in pred_indices]

print(df_pred.head(10))